# LowBitSparse Colab 验证手册

用于在 Colab A100 上复现 M0/M1/M2: FP16 基线、RTN/GPTQ/AWQ 权重量化、稀疏注意力长序列基准、WikiText-2 PPL、理论压缩比和汇总表。


## 推荐执行顺序

1. 环境检查和依赖安装。
2. 本地逻辑验证: `pytest` + `cpu_smoke.py`。
3. 快速单点验证: RTN INT4 只评少量样本。
4. M1 核心验收: baseline + INT8 + INT4 + GPTQ + AWQ。
5. M2 核心验收: Sliding Window + StreamingLLM 长序列基准。
6. 可选完整消融: `scripts/run_sweep.py`。


In [ ]:
# 1) 确认 GPU。M1 正式评测建议使用 A100。
!nvidia-smi -L

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [2]:
# 2) 挂载 Google Drive。结果写在 Drive 项目目录下,可避免 Colab 断连丢失。
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 3) 获取代码。若 Drive 中已存在仓库,直接进入;否则 clone 一份。
%cd /content/drive/MyDrive/LowBitSparse/

/content/drive/MyDrive/LowBitSparse


如果你确认 Drive 中没有未提交改动,可以手动执行下一格更新代码。若 `git status --short` 有本地改动,先不要 pull。

In [ ]:
# 可选:更新仓库。若有本地改动导致失败,先保留现场再处理。
# !git pull --ff-only

In [4]:
# 4) 安装依赖。Colab 自带 torch,requirements 只做宽松约束。
%pip install -q -r requirements.txt

import torch, transformers, datasets, accelerate
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 156.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 10.7 MB/s eta 0:00:00
torch: 2.11.0+cu128
transformers: 5.13.1
datasets: 4.0.0
accelerate: 1.14.0


In [ ]:
# 5) 可选:从 Colab Secrets 读取 Hugging Face token。
# 公开模型通常不需要 token;若遇到下载限流或私有模型,在 Secrets 中添加 HF_TOKEN。
import os
from google.colab import userdata

try:
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF_TOKEN 已设置")
else:
    print("未设置 HF_TOKEN,将按公开下载流程执行")

## 本地逻辑验证

这一步不下载 Qwen,用于确认量化数学、校准 hook、Linear 替换和压缩统计没有被改坏。

In [ ]:
!python -m pytest -q
!python scripts/cpu_smoke.py

## 快速单点验证

只跑 RTN INT4,并把 PPL 评测限制到 4 个窗口。适合验证 Colab 环境、模型下载、量化和结果落盘。

In [ ]:
# 生成一个临时 smoke config,不修改正式配置文件。
import copy
import yaml
from pathlib import Path

base = yaml.safe_load(Path("configs/qwen0.5b_int4.yaml").read_text(encoding="utf-8"))
smoke = copy.deepcopy(base)
smoke["exp_id"] = "m1_rtn_int4_g128_smoke"
smoke["eval"]["max_samples"] = 4
Path("configs/_colab_smoke_int4.yaml").write_text(
    yaml.safe_dump(smoke, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)
print(Path("configs/_colab_smoke_int4.yaml").read_text(encoding="utf-8"))

In [ ]:
!python main.py quant --config configs/_colab_smoke_int4.yaml

## M1 核心验收

这组命令生成 M1 最关键的对比: FP16 baseline、RTN INT8、RTN INT4、GPTQ INT4、AWQ INT4。

In [ ]:
!python main.py eval  --config configs/qwen0.5b_base.yaml
!python main.py quant --config configs/qwen0.5b_int8.yaml
!python main.py quant --config configs/qwen0.5b_int4.yaml
!python main.py quant --config configs/qwen0.5b_gptq_int4.yaml
!python main.py quant --config configs/qwen0.5b_awq_int4.yaml

!python scripts/summarize.py --results results --out results/m1_summary.md

In [ ]:
!python main.py quant --config configs/qwen0.5b_gptq_int4_embint8.yaml
!python main.py quant --config configs/qwen0.5b_gptq_int4_embint4.yaml

In [ ]:
# 查看汇总表。重点看 PPL、Delta PPL、压缩比和等效 bit。
from pathlib import Path
print(Path("results/m1_summary.md").read_text(encoding="utf-8"))

## 可选:完整 M1 消融

`run_sweep.py` 会扫描 method × bit × group_size。耗时明显长于核心验收,适合最终补表。先用 `--smoke` 确认流程,再打开完整 sweep。

In [ ]:
# 可选:全组合冒烟。每组只评 4 个 PPL 窗口,但 GPTQ/AWQ 仍会做校准。
# !python scripts/run_sweep.py --smoke
# !python scripts/summarize.py --results results --out results/m1_summary.md

In [ ]:
# 可选:完整消融。确认时间和额度足够后再执行。
RUN_FULL_SWEEP = True

if RUN_FULL_SWEEP:
    !python scripts/run_sweep.py
    !python scripts/summarize.py --results results --out results/m1_summary.md
else:
    print("RUN_FULL_SWEEP=False,已跳过完整消融")

## 补跑 GPTQ/AWQ per-channel。

重新汇总,新增的 m1_gptq_int4_gpc / m1_awq_int4_gpc 会并入表格。

In [ ]:
# 补跑 GPTQ/AWQ per-channel。复用 run_sweep.run_one,exp_id 自动为 m1_{method}_int4_gpc。
import sys, traceback
sys.path.insert(0, "scripts")
from scripts.run_sweep import run_one
from lowbitsparse.utils import load_config

base_cfg = load_config("configs/qwen0.5b_int4.yaml")
for method in ("gptq", "awq"):
    try:
        run_one(base_cfg, method, n_bits=4, group_size=-1, sym=False, smoke=False)
    except Exception:
        print(f"[失败] {method} per-channel:\n{traceback.format_exc()}")

# 重新汇总,新增的 m1_gptq_int4_gpc / m1_awq_int4_gpc 会并入表格。
!python scripts/summarize.py --results results --out results/m1_summary.md
from pathlib import Path
print(Path("results/m1_summary.md").read_text(encoding="utf-8"))

## 重跑 AWQ:恢复无裁剪好值(M1-h 负结果)

AWQ 的裁剪搜索(auto_clip)经 A100 实测**全面恶化 PPL**(INT3 +17),已默认关闭(见 OPTIMIZATION M1-h)。当前 `results/m1_awq_*.json` 是裁剪的坏值,下一格用默认(裁剪关)重跑 5 个 AWQ 组合恢复正确结果。

In [ ]:
# 重跑 AWQ 全组合(裁剪现默认关),覆盖裁剪坏值,恢复无裁剪好值
!python scripts/run_sweep.py --only awq --no-emb
!python scripts/summarize.py --results results --out results/m1_summary.md

from pathlib import Path
print(Path("results/m1_summary.md").read_text(encoding="utf-8"))

## M2 稀疏注意力

这组命令跑 Sliding Window 和 StreamingLLM 的长序列对照。`block_sparse` 作为可选实验保留在配置里，确认时间和额度足够再开。


In [ ]:
# M2 核心验收: baseline vs sparse 长序列基准。
!python main.py sparse --config configs/qwen0.5b_sparse_sliding.yaml
!python main.py sparse --config configs/qwen0.5b_sparse_streaming.yaml

# 可选:块稀疏
!python main.py sparse --config configs/qwen0.5b_sparse_block.yaml


## M2-c StreamingLLM KV cache 裁剪

这一步在 M2-b 的 StreamingLLM 上做真实 KV cache 裁剪。质量仍看 mask 参考,decode 阶段则走 sink + window 的短 cache 路径。


In [ ]:
# M2-c:StreamingLLM KV cache pruning.
!python main.py sparse --config configs/qwen0.5b_sparse_streaming_kvprune.yaml


In [ ]:
!python main.py sparse --config configs/qwen0.5b_sparse_streaming_compile.yaml

config.json: 100% 659/659 [00:00<00:00, 3.98MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 10.9MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 98.7MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 101MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 119MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   1% 14.3M/988M [00:01<01:18, 12.4MB/s, 6.24kB/s  ]
model.safetensors: downloading bytes:  40% 399M/988M [00:02<00:01, 414MB/s, 31.7MB/s  ]
model.safetensors: downloading bytes:  45% 449M/988M [00:02<00:01, 437MB/s, 36.4MB/s  ]
model.safetensors: downloading bytes:  86% 854M/988M [00:04<00:00, 246MB/s, 71.4MB/s  ]
model.safetensors: downloading bytes: 100% 854M/854M [00:04<00:00, 174MB/s, 71.1MB/s  ]
model.safetensors: reconstructing file: 100% 988M/988M [00:04<00:00, 201MB/s, 85.4MB/s  ]
Loading weights: 100% 290/290 [00:00<00:00, 1517.68it/s]
generation_config.json: 100% 242/242 [00:00<00:00, 1.65MB/s]
[14:08:58

In [ ]:
# 查看 M2 结果文件和逐长度对照。
import json
from pathlib import Path

for path in sorted(Path("results").glob("m2_*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    print(f"== {path.name} ==")
    for row in data.get("rows", []):
        print(
            row["seqlen"],
            "prefill_speedup=", row["speedup"]["prefill"],
            "decode_speedup=", row["speedup"]["decode"],
            "baseline_ppl=", row["baseline"]["ppl"]["ppl"],
            "sparse_ppl=", row["sparse"]["ppl"]["ppl"],
        )


## 结果备份

如果项目目录已经在 Drive 中,结果天然保存在 Drive。下面这格会额外复制一份带时间戳的 results 快照,避免覆盖历史实验。

In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

src = Path("results").resolve()
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
dst = Path("/content/drive/MyDrive/LowBitSparse_results") / stamp
dst.parent.mkdir(parents=True, exist_ok=True)

if src.exists():
    shutil.copytree(src, dst)
    print("results 快照已保存到:", dst)
else:
    print("results 目录不存在,没有可备份内容")